# Notebook 04 — Modelos Avançados (LSTM + BERT/Bertimbau) · Movies

Replica as **5 visões** do notebook 03 com modelos de **deep learning**:

- **LSTM** (PyTorch) — embeddings aprendidos do zero
- **BERT (Bertimbau base)** — `neuralmind/bert-base-portuguese-cased`, fine-tuning

Objetivo: contrastar **técnicas básicas** (NB/LR/SVM do notebook 03) com **técnicas avançadas** para mostrar se o reality gap (V3/V5) persiste com modelos contextuais.

## ✨ Auto-setup

A **Célula 1** detecta o ambiente e cuida das dependências automaticamente:
- **No Colab:** instala silenciosamente `transformers`, `accelerate`, `kagglehub` (PyTorch+CUDA já vem pronto)
- **Local:** verifica se PyTorch tem CUDA habilitado e mostra o comando de instalação se faltar

## ⏱️ Tempo esperado (RTX 3070, 8GB VRAM, dataset real completo ~1.5M)

| Etapa | Tempo |
|---|---|
| LSTM em V1/V2/V3/V5 (visões pequenas) | ~5 min total |
| LSTM em V4 (1.2M frases) | ~30-60 min |
| BERT em V1/V2/V3/V5 (visões pequenas) | ~10 min total |
| BERT em V4 (1.2M frases, 3 epochs) | **~6-9 HORAS** ⚠️ |
| **Total (rodando overnight)** | **~7-10 horas** |

Para teste rápido, ajuste `MAX_SAMPLES_REAL = 100_000` na **Célula 3**.

In [2]:
# ============================================================
# CÉLULA 1 — AUTO-SETUP DO AMBIENTE (GPU first, com bootstrap local se necessário)
# ============================================================
import sys, subprocess, importlib.util, os, shutil

IN_COLAB = 'google.colab' in sys.modules
print(f'Ambiente detectado: {"Google Colab" if IN_COLAB else "Local"}')
print(f'Python: {sys.version.split()[0]} ({sys.executable})')

# Configurações
CUDA_CANDIDATES = ['cu121', 'cu118']
AUTO_INSTALL_PYTORCH_CUDA = True
BOOTSTRAP_PY312_WITH_UV = True
ALLOW_CPU_FALLBACK = False
UV_ENV_DIR = '.venv-cu121-py312'
UV_KERNEL_NAME = 'tcc-cu121-py312'
UV_KERNEL_DISPLAY = 'Python 3.12 (TCC CUDA)'

def _run(cmd, check=True, quiet=False):
    """Executa comando shell; em quiet captura stdout/stderr para diagnóstico."""
    if quiet:
        return subprocess.run(cmd, check=check, capture_output=True, text=True)
    return subprocess.run(cmd, check=check)

def _format_cmd(cmd):
    return ' '.join(str(x) for x in cmd)

def _tail(text, lines=20):
    if not text:
        return ''
    partes = text.strip().splitlines()
    return '\n'.join(partes[-lines:])

def _print_subprocess_error(e):
    stdout = getattr(e, 'stdout', None) or getattr(e, 'output', '') or ''
    stderr = getattr(e, 'stderr', '') or ''
    if stdout:
        print('\n--- stdout ---')
        print(stdout)
    if stderr:
        print('\n--- stderr ---')
        print(stderr)

def _repo_dir():
    cwd = os.getcwd()
    if os.path.basename(cwd).lower() == 'src':
        return os.path.abspath(os.path.join(cwd, '..'))
    return cwd

def _module_existe(name):
    """Verifica se módulo está instalado sem importá-lo no kernel atual."""
    return importlib.util.find_spec(name) is not None

def _torch_info_for_python(python_exe):
    """Consulta torch em subprocesso para não contaminar o kernel atual."""
    code = r'''
import importlib.util
spec = importlib.util.find_spec("torch")
if spec is None:
    print("MISSING")
else:
    import torch
    print("OK")
    print(torch.__version__)
    print(torch.cuda.is_available())
'''
    try:
        r = subprocess.run(
            [python_exe, '-c', code],
            capture_output=True, text=True, timeout=60, check=True
        )
        linhas = r.stdout.strip().splitlines()
        if not linhas or linhas[0].strip() == 'MISSING':
            return None
        versao = linhas[1].strip() if len(linhas) > 1 else 'desconhecida'
        cuda_ok = len(linhas) > 2 and linhas[2].strip() == 'True'
        return {'version': versao, 'cuda': cuda_ok}
    except Exception:
        return {'version': 'desconhecida', 'cuda': False}

def _torch_info():
    if not _module_existe('torch'):
        return None
    return _torch_info_for_python(sys.executable)

def _tem_nvidia_smi():
    try:
        r = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=20)
        return r.returncode == 0
    except Exception:
        return False

def _python_windows_compativel_com_torch_cuda():
    return (3, 9) <= sys.version_info[:2] <= (3, 12)

def _bootstrap_local_py312_kernel():
    uv_exe = shutil.which('uv')
    if uv_exe is None:
        raise SystemExit('uv não encontrado. Instale o uv ou rode este notebook em Python 3.12.')

    repo_dir = _repo_dir()
    env_dir = os.path.join(repo_dir, UV_ENV_DIR)
    env_python = os.path.join(env_dir, 'Scripts', 'python.exe')

    if os.path.exists(env_python):
        info_existente = _torch_info_for_python(env_python)
        if info_existente and info_existente['cuda']:
            print(f'\nAmbiente Python 3.12 já existe e já está com CUDA ativa ({info_existente["version"]}).')
            print('Registrando/atualizando kernel Jupyter desse ambiente...')
            print('CMD:', _format_cmd([env_python, '-m', 'ipykernel', 'install', '--user', '--name', UV_KERNEL_NAME,
                  '--display-name', UV_KERNEL_DISPLAY]))
            try:
                _run([env_python, '-m', 'ipykernel', 'install', '--user', '--name', UV_KERNEL_NAME,
                      '--display-name', UV_KERNEL_DISPLAY], quiet=True)
            except subprocess.CalledProcessError as e:
                _print_subprocess_error(e)
                raise SystemExit('Falha ao registrar o kernel Jupyter Python 3.12 (TCC CUDA).')
            raise SystemExit(
                f'Kernel CUDA já estava pronto. Troque o kernel do notebook para "{UV_KERNEL_DISPLAY}" e rode a Célula 1 novamente.'
            )
        else:
            print('\nAmbiente Python 3.12 já existe, mas ainda não está pronto para CUDA.')
            print('Ele será recriado do zero com --clear.')

    etapas = [
        ('Baixando Python 3.12 via uv', [uv_exe, 'python', 'install', '3.12']),
        ('Criando ambiente virtual 3.12', [uv_exe, 'venv', '--clear', '--python', '3.12', env_dir]),
    ]
    for titulo, cmd in etapas:
        print(f'\n{titulo}...')
        print('CMD:', _format_cmd(cmd))
        try:
            _run(cmd, quiet=True)
        except subprocess.CalledProcessError as e:
            _print_subprocess_error(e)
            raise SystemExit(f'Falha em: {titulo}.')

    pacotes_base = [
        'ipykernel', 'numpy', 'pandas', 'scikit-learn',
        'matplotlib', 'kagglehub>=1.0.1', 'transformers>=4.40', 'accelerate>=0.30', 'tqdm>=4.65'
    ]
    print('\nInstalando dependências base no ambiente Python 3.12...')
    print('CMD:', _format_cmd([uv_exe, 'pip', 'install', '--python', env_python] + pacotes_base))
    try:
        _run([uv_exe, 'pip', 'install', '--python', env_python] + pacotes_base, quiet=True)
    except subprocess.CalledProcessError as e:
        _print_subprocess_error(e)
        raise SystemExit('Falha ao instalar dependências base no ambiente Python 3.12.')

    torch_ok = False
    for cuda_version in CUDA_CANDIDATES:
        print(f'\nInstalando PyTorch CUDA ({cuda_version}) no ambiente Python 3.12...')
        cmd = [uv_exe, 'pip', 'install', '--python', env_python, '--upgrade', '--reinstall', 'torch',
               '--index-url', f'https://download.pytorch.org/whl/{cuda_version}']
        print('CMD:', _format_cmd(cmd))
        try:
            _run(cmd, quiet=True)
        except subprocess.CalledProcessError as e:
            _print_subprocess_error(e)
            continue
        info_env = _torch_info_for_python(env_python)
        if info_env and info_env['cuda']:
            torch_ok = True
            print(f'OK: ambiente auxiliar pronto com {info_env["version"]} e CUDA ativa.')
            break

    if not torch_ok:
        raise SystemExit('O ambiente Python 3.12 foi criado, mas o PyTorch CUDA não ativou nele.')

    print('\nRegistrando kernel Jupyter do ambiente CUDA...')
    print('CMD:', _format_cmd([env_python, '-m', 'ipykernel', 'install', '--user', '--name', UV_KERNEL_NAME,
          '--display-name', UV_KERNEL_DISPLAY]))
    try:
        _run([env_python, '-m', 'ipykernel', 'install', '--user', '--name', UV_KERNEL_NAME,
              '--display-name', UV_KERNEL_DISPLAY], quiet=True)
    except subprocess.CalledProcessError as e:
        _print_subprocess_error(e)
        raise SystemExit('Falha ao registrar o kernel Jupyter Python 3.12 (TCC CUDA).')

    raise SystemExit(
        f'Kernel CUDA preparado com sucesso. Troque o kernel do notebook para "{UV_KERNEL_DISPLAY}" e rode a Célula 1 novamente.'
    )

# ============================================================
# 1) DEPENDÊNCIAS LEVES (transformers, accelerate, kagglehub, tqdm)
# ============================================================
pkgs_simples = []
for pkg, name in [('transformers>=4.40', 'transformers'),
                  ('accelerate>=0.30', 'accelerate'),
                  ('kagglehub>=1.0.1', 'kagglehub'),
                  ('tqdm>=4.65', 'tqdm')]:
    if not _module_existe(name):
        pkgs_simples.append(pkg)

if pkgs_simples:
    print(f'\nInstalando dependências leves: {pkgs_simples}')
    _run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs_simples)
else:
    print('Dependências leves já presentes (transformers, accelerate, kagglehub, tqdm).')

# ============================================================
# 2) PYTORCH COM CUDA
# ============================================================
if IN_COLAB:
    print('\nColab: PyTorch + CUDA já vêm pré-instalados, sem ação necessária.')
else:
    if _tem_nvidia_smi():
        print('nvidia-smi respondeu: existe GPU NVIDIA/driver visível no sistema.')
    else:
        print('nvidia-smi não respondeu. Se houver GPU NVIDIA, confira driver e PATH.')

    if not _python_windows_compativel_com_torch_cuda():
        print('\nAVISO: o kernel atual está em uma versão de Python fora da janela suportada pelo PyTorch CUDA no Windows.')
        print('Para Windows, o fluxo oficial costuma suportar Python 3.9 a 3.12.')
        if BOOTSTRAP_PY312_WITH_UV:
            _bootstrap_local_py312_kernel()
        raise SystemExit('Rode este notebook com Python 3.12 para usar CUDA localmente.')

    torch_info = _torch_info()
    if torch_info and torch_info['cuda']:
        print(f"\nOK: PyTorch com CUDA já instalado ({torch_info['version']}).")
    else:
        if torch_info is None:
            motivo = 'PyTorch não está instalado'
        else:
            motivo = f"PyTorch instalado sem CUDA (versão atual: {torch_info['version']})"
        print(f'\nAVISO: {motivo}.')

        if 'torch' in sys.modules and AUTO_INSTALL_PYTORCH_CUDA:
            print('\nERRO: torch já foi importado neste kernel. Reinicie o kernel antes de reinstalar.')
            raise SystemExit('Reinicie o kernel manualmente.')

        if AUTO_INSTALL_PYTORCH_CUDA:
            instalou_cuda = False
            for cuda_version in CUDA_CANDIDATES:
                print(f'\nInstalando PyTorch com CUDA ({cuda_version})...')
                cmd_cuda = [sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall',
                            'torch', '--index-url', f'https://download.pytorch.org/whl/{cuda_version}']
                if torch_info is None:
                    cmd_cuda = [sys.executable, '-m', 'pip', 'install', 'torch',
                                '--index-url', f'https://download.pytorch.org/whl/{cuda_version}']
                try:
                    _run(cmd_cuda, quiet=True)
                except subprocess.CalledProcessError as e:
                    print(_tail(getattr(e, 'stderr', '') or getattr(e, 'output', '') or ''))
                    continue

                torch_info = _torch_info()
                if torch_info and torch_info['cuda']:
                    instalou_cuda = True
                    print(f"OK: PyTorch com CUDA instalado com sucesso ({torch_info['version']}).")
                    break

            if not instalou_cuda:
                if torch_info is None:
                    raise SystemExit('Torch não ficou disponível após a tentativa de instalação.')
                if ALLOW_CPU_FALLBACK:
                    print('\nAVISO: nenhuma tentativa de instalação CUDA ativou a GPU. Seguindo com torch em CPU.')
                else:
                    raise SystemExit('Falha em habilitar CUDA automaticamente neste kernel após tentar: ' + ', '.join(CUDA_CANDIDATES))
        else:
            if torch_info is None:
                raise SystemExit('Torch não encontrado. Ative AUTO_INSTALL_PYTORCH_CUDA ou instale manualmente.')
            if ALLOW_CPU_FALLBACK:
                print('\nSeguindo com a versão atual de torch em CPU. LSTM roda; BERT pode ficar inviável localmente.')
            else:
                raise SystemExit('Sem CUDA e fallback em CPU desativado.')

# ============================================================
# 3) AGORA SIM, importa torch
# ============================================================
import torch
print(f'\nPyTorch versão: {torch.__version__}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'OK: CUDA disponível — GPU: {gpu_name} ({gpu_vram:.1f} GB VRAM)')
else:
    if ALLOW_CPU_FALLBACK:
        print('AVISO: sem CUDA — LSTM roda em CPU, mas BERT pode ficar inviável no ambiente local. Use Colab GPU se necessário.')
    else:
        raise SystemExit('PROVA DE FOGO FALHOU: torch foi importado, mas torch.cuda.is_available() retornou False.')


Ambiente detectado: Local
Python: 3.12.9 (c:\Users\Davi\Documentos\tcc-analise-sentimento\.venv-cu121-py312\Scripts\python.exe)
Dependências leves já presentes (transformers, accelerate, kagglehub, tqdm).
nvidia-smi respondeu: existe GPU NVIDIA/driver visível no sistema.

OK: PyTorch com CUDA já instalado (2.5.1+cu121).

PyTorch versão: 2.5.1+cu121
OK: CUDA disponível — GPU: NVIDIA GeForce RTX 3070 (8.6 GB VRAM)


In [3]:
# ============================================================
# CÉLULA 2 — Imports principais + paths + reprodutibilidade
# ============================================================
import os, shutil, re, time, json
import numpy as np
import pandas as pd
import torch

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paths (Colab clona o repo, local usa estrutura existente)
if IN_COLAB:
    if not os.path.exists('/content/tcc-analise-sentimento'):
        os.system('git clone https://github.com/ROMAUSKI/tcc-analise-sentimento.git /content/tcc-analise-sentimento')
    else:
        os.system('cd /content/tcc-analise-sentimento && git pull')
    BASE_DIR = '/content/tcc-analise-sentimento'
    REAL_DATA_PATH = '/content/sample_data/dataset_real_sentimentos'
else:
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
    REAL_DATA_PATH = os.path.join(BASE_DIR, 'src', 'dados_reais')

# Dataset real via kagglehub se ainda não existir
if not os.path.exists(REAL_DATA_PATH) or not os.listdir(REAL_DATA_PATH):
    import kagglehub
    print('Baixando dataset real do Kaggle (~877 MB, só na 1ª vez)...')
    cache_path = kagglehub.dataset_download('fredericods/ptbr-sentiment-analysis-datasets')
    os.makedirs(REAL_DATA_PATH, exist_ok=True)
    for f in os.listdir(cache_path):
        shutil.copy(os.path.join(cache_path, f), os.path.join(REAL_DATA_PATH, f))
    print(f'Dataset real disponível em: {REAL_DATA_PATH}')
else:
    print(f'Dataset real já existe em: {REAL_DATA_PATH}')

DADOS_BRUTOS_MOVIES = os.path.join(BASE_DIR, 'dados', 'brutos')
DADOS_BRUTOS_APPS   = os.path.join(BASE_DIR, 'dados', 'brutos_apps')
DADOS_PROCESSADO    = os.path.join(BASE_DIR, 'dados', 'processado')
RESULTADOS          = os.path.join(BASE_DIR, 'resultados')
CHECKPOINTS         = os.path.join(BASE_DIR, 'src', 'checkpoints_avancado')
os.makedirs(CHECKPOINTS, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nBASE_DIR: {BASE_DIR}')
print(f'DEVICE:   {DEVICE}')


Dataset real já existe em: c:\Users\Davi\Documentos\tcc-analise-sentimento\src\dados_reais

BASE_DIR: c:\Users\Davi\Documentos\tcc-analise-sentimento
DEVICE:   cuda


## Configurações principais

Ajuste estas constantes **antes** de executar. Ordem de impacto no tempo total:

1. `MAX_SAMPLES_REAL` — `None` para tudo (~1.5M, demora horas), `100_000` para teste rápido
2. `BERT_EPOCHS` — 3 é o padrão da literatura, 2 corta tempo pela metade
3. `BERT_BATCH_SIZE` — aumentar para acelerar (até estourar VRAM); diminuir se OOM
4. `LSTM_EPOCHS` — 5 é OK; mais que 10 tende a overfitar com 480 frases (V1/V2/V3)

In [4]:
# ============================================================
# CÉLULA 3 — CONSTANTES (ajuste conforme tempo disponível)
# ============================================================

# --- Volume do dataset real (V4 e V5 usam) ---
MAX_SAMPLES_REAL = None         # None = TODOS (~1.5M, demora HORAS); 100_000 = teste rápido

# --- LSTM ---
LSTM_EMBEDDING_DIM = 128
LSTM_HIDDEN_DIM    = 128
LSTM_EPOCHS        = 5
LSTM_BATCH_SIZE    = 32
LSTM_LR            = 1e-3
LSTM_DROPOUT       = 0.3
LSTM_MAX_LEN       = 100   # tokens por frase (truncar/padding)

# --- BERT (Bertimbau base) ---
BERT_MODEL_NAME    = 'neuralmind/bert-base-portuguese-cased'
BERT_MODEL_REVISION = '4a78cfbf83c9c97533dd6d6694ca4323029ff061'  # revisão com model.safetensors
BERT_EPOCHS        = 3
BERT_BATCH_SIZE    = 16        # se OOM na 3070, reduzir para 8
BERT_MAX_LENGTH    = 128       # tokens (subwords) por frase
BERT_LR            = 2e-5
BERT_WEIGHT_DECAY  = 0.01

# --- Sintético (usar TUDO: 600/classe = 1800 total) ---
N_POR_CLASSE_SINT = 600

print('Configurações:')
print(f'  MAX_SAMPLES_REAL: {MAX_SAMPLES_REAL if MAX_SAMPLES_REAL else "TODOS (~1.5M)"}')
print(f'  BERT: {BERT_MODEL_NAME} @ {BERT_MODEL_REVISION[:7]}')
print(f'  BERT epochs={BERT_EPOCHS}, batch={BERT_BATCH_SIZE}, max_len={BERT_MAX_LENGTH}')
print(f'  LSTM epochs={LSTM_EPOCHS}, batch={LSTM_BATCH_SIZE}, hidden={LSTM_HIDDEN_DIM}')


Configurações:
  MAX_SAMPLES_REAL: TODOS (~1.5M)
  BERT: neuralmind/bert-base-portuguese-cased
  BERT epochs=3, batch=16, max_len=128
  LSTM epochs=5, batch=32, hidden=128


## 1. Carregar dados (sintético + real balanceado + real desbalanceado) e preparar splits

Reusa exatamente a mesma lógica do notebook 03 para garantir comparabilidade dos resultados.

In [5]:
# ============================================================
# CÉLULA 5 — Carregar dados (mesmas regras do notebook 03)
# ============================================================
from sklearn.model_selection import train_test_split

def limpar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r'[^a-zà-ÿ\s]', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# --- Sintético (200/classe) ---
df_sintetico = pd.read_csv(os.path.join(DADOS_PROCESSADO, 'synthetic_dataset.csv'))
df_sintetico = df_sintetico[['frase_limpa', 'classe', 'fonte']].dropna()
df_sintetico_bal = (df_sintetico.groupby('classe')
                                .sample(n=N_POR_CLASSE_SINT, random_state=SEED)
                                .reset_index(drop=True))
print(f'Sintético: {len(df_sintetico_bal)} frases ({N_POR_CLASSE_SINT}/classe)')

# --- Real ---
path_real = os.path.join(REAL_DATA_PATH, 'utlc_movies.csv')
print(f'\nCarregando dataset real (MAX_SAMPLES_REAL={MAX_SAMPLES_REAL})...')
t0 = time.time()
df_real_raw = pd.read_csv(path_real, nrows=MAX_SAMPLES_REAL)
print(f'  {len(df_real_raw):,} linhas carregadas em {time.time()-t0:.1f}s')

def rating_para_classe(rating):
    if rating >= 4: return 'Positiva'
    elif rating <= 2: return 'Negativa'
    else: return 'Neutra'

df_real_raw['classe'] = df_real_raw['rating'].apply(rating_para_classe)
df_real = df_real_raw[['review_text', 'classe']].dropna().rename(columns={'review_text': 'frase'})
df_real['frase_limpa'] = df_real['frase'].apply(limpar_texto)
df_real = df_real[df_real['frase_limpa'] != ''].reset_index(drop=True)
print(f'Real (após limpeza): {len(df_real):,} frases')

# --- Real BALANCEADO (200/classe) ---
df_real_bal = (df_real.groupby('classe')
                      .sample(n=N_POR_CLASSE_SINT, random_state=SEED)
                      .reset_index(drop=True))
print(f'Real balanceado: {len(df_real_bal)} frases ({N_POR_CLASSE_SINT}/classe)')

# --- Real DESBALANCEADO (todos) ---
df_real_desbal = df_real.copy().reset_index(drop=True)
print(f'Real desbalanceado: {len(df_real_desbal):,} frases')
print(df_real_desbal['classe'].value_counts(normalize=True).round(3))

# --- Splits 80/20 estratificados (5 visões) ---
S_tr, S_te = train_test_split(df_sintetico_bal, test_size=0.2, stratify=df_sintetico_bal['classe'], random_state=SEED)
R_tr, R_te = train_test_split(df_real_bal,      test_size=0.2, stratify=df_real_bal['classe'],      random_state=SEED)
RD_tr, RD_te = train_test_split(df_real_desbal, test_size=0.2, stratify=df_real_desbal['classe'], random_state=SEED)

print(f'\nSplits:')
print(f'  V1: Sintético→Sintético — {len(S_tr)} treino / {len(S_te)} teste')
print(f'  V2: Real(bal)→Real(bal) — {len(R_tr)} treino / {len(R_te)} teste')
print(f'  V3: Sintético→Real(bal) — usa S_tr para treino e R_te para teste')
print(f'  V4: Real(desbal)→Real(desbal) — {len(RD_tr):,} treino / {len(RD_te):,} teste')
print(f'  V5: Sintético→Real(desbal) — usa S_tr para treino e RD_te para teste')


Sintético: 600 frases (200/classe)

Carregando dataset real (MAX_SAMPLES_REAL=None)...
  1,487,449 linhas carregadas em 11.6s
Real (após limpeza): 1,483,927 frases
Real balanceado: 600 frases (200/classe)
Real desbalanceado: 1,483,927 frases
classe
Positiva    0.707
Neutra      0.200
Negativa    0.093
Name: proportion, dtype: float64

Splits:
  V1: Sintético→Sintético — 480 treino / 120 teste
  V2: Real(bal)→Real(bal) — 480 treino / 120 teste
  V3: Sintético→Real(bal) — usa S_tr para treino e R_te para teste
  V4: Real(desbal)→Real(desbal) — 1,187,141 treino / 296,786 teste
  V5: Sintético→Real(desbal) — usa S_tr para treino e RD_te para teste


## 2. LSTM — definição e treino

Embeddings aprendidos do zero (sem pré-treino). 1 camada LSTM unidirecional.
Vocabulário **construído no treino de cada visão** (palavras não vistas viram `<UNK>`).

In [6]:
# ============================================================
# CÉLULA 7 — LSTM: arquitetura + helpers de tokenização
# ============================================================
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

PAD, UNK = 0, 1

def tokenize(texto):
    """Tokenização simples por palavra (split em espaço)"""
    return texto.split()

def build_vocab(textos, min_freq=1, max_size=20000):
    """Constrói vocabulário com índices reservados PAD=0, UNK=1"""
    counter = Counter()
    for t in textos:
        counter.update(tokenize(t))
    vocab = {'<PAD>': PAD, '<UNK>': UNK}
    for word, freq in counter.most_common(max_size - 2):
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

def encode(texto, vocab, max_len):
    """Converte texto em lista de índices, padded/truncated para max_len"""
    ids = [vocab.get(w, UNK) for w in tokenize(texto)[:max_len]]
    length = len(ids)
    ids = ids + [PAD] * (max_len - length)
    return ids, length

class TextDataset(Dataset):
    def __init__(self, textos, labels, vocab, max_len):
        self.data = [encode(t, vocab, max_len) for t in textos]
        self.labels = labels.values if hasattr(labels, 'values') else list(labels)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        ids, length = self.data[idx]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(length, dtype=torch.long), self.labels[idx]

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h = self.dropout(h.squeeze(0))
        return self.fc(h)

print('LSTM definido. Pronto para treino.')


LSTM definido. Pronto para treino.


In [7]:
# ============================================================
# CÉLULA 8 — Função run_lstm_vision(X_tr, y_tr, X_te, y_te, visao_nome)
# ============================================================
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score

def run_lstm_vision(X_tr, y_tr, X_te, y_te, visao_nome):
    print(f'\n=== LSTM | {visao_nome} ===')
    t0 = time.time()
    le = LabelEncoder()
    y_tr_enc = le.fit_transform(y_tr)
    y_te_enc = le.transform(y_te)
    num_classes = len(le.classes_)

    vocab = build_vocab(X_tr, min_freq=1, max_size=20000)
    print(f'  Vocabulário: {len(vocab):,} tokens')

    train_ds = TextDataset(X_tr.tolist() if hasattr(X_tr, "tolist") else list(X_tr),
                            pd.Series(y_tr_enc), vocab, LSTM_MAX_LEN)
    test_ds  = TextDataset(X_te.tolist() if hasattr(X_te, "tolist") else list(X_te),
                            pd.Series(y_te_enc), vocab, LSTM_MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=LSTM_BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=LSTM_BATCH_SIZE, shuffle=False)

    model = LSTMClassifier(len(vocab), LSTM_EMBEDDING_DIM, LSTM_HIDDEN_DIM, num_classes, LSTM_DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LSTM_LR)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(LSTM_EPOCHS):
        model.train()
        total_loss = 0
        for ids, lengths, labels in train_loader:
            ids, lengths, labels = ids.to(DEVICE), lengths.to(DEVICE), labels.to(DEVICE).long()
            optimizer.zero_grad()
            logits = model(ids, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
        print(f'  Epoch {epoch+1}/{LSTM_EPOCHS} | loss={total_loss/len(train_ds):.4f}')

    # Avaliação
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for ids, lengths, labels in test_loader:
            ids, lengths = ids.to(DEVICE), lengths.to(DEVICE)
            logits = model(ids, lengths)
            preds.extend(logits.argmax(dim=1).cpu().numpy())
            trues.extend(labels.numpy())

    acc = accuracy_score(trues, preds)
    prec, rec, f1w, _ = precision_recall_fscore_support(trues, preds, average='weighted', zero_division=0)
    f1m = f1_score(trues, preds, average='macro', zero_division=0)
    elapsed = time.time() - t0
    print(f'  ✓ LSTM | F1w={f1w:.2%}  F1macro={f1m:.2%}  Acc={acc:.2%}  ({elapsed:.0f}s)')

    return {'Visão': visao_nome, 'Modelo': 'LSTM',
            'Acurácia': acc, 'Precisão': prec, 'Recall': rec,
            'F1-Score': f1w, 'F1-Macro': f1m, 'Tempo (s)': round(elapsed, 1)}


## 3. BERT (Bertimbau) — fine-tuning

Usa `neuralmind/bert-base-portuguese-cased` (110M parâmetros, ~440MB de download na 1ª vez).
Fine-tuning supervisionado com `transformers.Trainer`.

In [8]:
# ============================================================
# CÉLULA 10 — BERT: tokenizer + dataset
# ============================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding

print(f'Carregando tokenizer {BERT_MODEL_NAME} @ {BERT_MODEL_REVISION[:7]}...')
BERT_TOKENIZER = AutoTokenizer.from_pretrained(BERT_MODEL_NAME, revision=BERT_MODEL_REVISION)
print(f'  vocab_size: {BERT_TOKENIZER.vocab_size}, max_len recomendado: {BERT_TOKENIZER.model_max_length}')

class BertDataset(torch.utils.data.Dataset):
    def __init__(self, textos, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            textos.tolist() if hasattr(textos, "tolist") else list(textos),
            truncation=True, padding=False, max_length=max_length
        )
        self.labels = labels.values if hasattr(labels, 'values') else list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(int(self.labels[idx]))
        return item


c:\Users\Davi\Documentos\tcc-analise-sentimento\.venv-cu121-py312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Carregando tokenizer neuralmind/bert-base-portuguese-cased...
  vocab_size: 29794, max_len recomendado: 1000000000000000019884624838656


In [9]:
# ============================================================
# CÉLULA 11 — Função run_bert_vision(X_tr, y_tr, X_te, y_te, visao_nome)
# ============================================================
import warnings
warnings.filterwarnings('ignore')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1w, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    f1m = f1_score(labels, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1_weighted': f1w, 'f1_macro': f1m}

def run_bert_vision(X_tr, y_tr, X_te, y_te, visao_nome):
    print(f'\n=== BERT | {visao_nome} ===')
    print(f'  Treino: {len(X_tr):,} frases | Teste: {len(X_te):,} frases')
    t0 = time.time()

    le = LabelEncoder()
    y_tr_enc = le.fit_transform(y_tr)
    y_te_enc = le.transform(y_te)
    num_classes = len(le.classes_)

    train_ds = BertDataset(X_tr, pd.Series(y_tr_enc), BERT_TOKENIZER, BERT_MAX_LENGTH)
    test_ds  = BertDataset(X_te, pd.Series(y_te_enc), BERT_TOKENIZER, BERT_MAX_LENGTH)

    print(f'  Carregando pesos safetensors de {BERT_MODEL_NAME} @ {BERT_MODEL_REVISION[:7]}...')
    model = AutoModelForSequenceClassification.from_pretrained(
        BERT_MODEL_NAME,
        revision=BERT_MODEL_REVISION,
        use_safetensors=True,
        num_labels=num_classes,
    ).to(DEVICE)

    safe_name = re.sub(r'[^a-zA-Z0-9]+', '_', visao_nome).lower()
    out_dir = os.path.join(CHECKPOINTS, f'bert_{safe_name}')

    training_args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=BERT_EPOCHS,
        per_device_train_batch_size=BERT_BATCH_SIZE,
        per_device_eval_batch_size=BERT_BATCH_SIZE * 2,
        learning_rate=BERT_LR,
        weight_decay=BERT_WEIGHT_DECAY,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,                # mantém só o último checkpoint
        load_best_model_at_end=False,
        logging_steps=200,
        seed=SEED,
        report_to='none',
        fp16=torch.cuda.is_available(),    # mixed precision se GPU
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        tokenizer=BERT_TOKENIZER,
        data_collator=DataCollatorWithPadding(BERT_TOKENIZER),
        compute_metrics=compute_metrics,
    )

    trainer.train()
    final_metrics = trainer.evaluate()
    elapsed = time.time() - t0

    f1w = final_metrics['eval_f1_weighted']
    f1m = final_metrics['eval_f1_macro']
    acc = final_metrics['eval_accuracy']
    print(f'  ✓ BERT | F1w={f1w:.2%}  F1macro={f1m:.2%}  Acc={acc:.2%}  ({elapsed/60:.1f}min)')

    # Libera VRAM
    del model, trainer
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return {'Visão': visao_nome, 'Modelo': 'BERT (Bertimbau)',
            'Acurácia': acc, 'Precisão': final_metrics['eval_precision'],
            'Recall': final_metrics['eval_recall'],
            'F1-Score': f1w, 'F1-Macro': f1m, 'Tempo (s)': round(elapsed, 1)}


## 4. Executar todas as visões

Salva resultados parciais em `resultados/metricas_avancado_movies_partial.csv` a cada visão concluída — se travar, você não perde tudo.

In [10]:
# ============================================================
# CÉLULA 13 — LSTM nas 5 visões
# ============================================================
resultados_avancados = []
PARTIAL_CSV = os.path.join(RESULTADOS, 'metricas_avancado_movies_partial.csv')

def salvar_parcial():
    if resultados_avancados:
        pd.DataFrame(resultados_avancados).to_csv(PARTIAL_CSV, index=False)
        print(f'  💾 Parcial salvo: {PARTIAL_CSV}')

# V1: Sintético → Sintético
resultados_avancados.append(run_lstm_vision(
    S_tr['frase_limpa'], S_tr['classe'], S_te['frase_limpa'], S_te['classe'],
    'V1: Sintético → Sintético'))
salvar_parcial()

# V2: Real → Real (controlado)
resultados_avancados.append(run_lstm_vision(
    R_tr['frase_limpa'], R_tr['classe'], R_te['frase_limpa'], R_te['classe'],
    'V2: Real → Real'))
salvar_parcial()

# V3: Sintético → Real (controlado, mesmo test set do V2)
resultados_avancados.append(run_lstm_vision(
    S_tr['frase_limpa'], S_tr['classe'], R_te['frase_limpa'], R_te['classe'],
    'V3: Sintético → Real'))
salvar_parcial()

# V4: Real → Real (desbalanceado)
resultados_avancados.append(run_lstm_vision(
    RD_tr['frase_limpa'], RD_tr['classe'], RD_te['frase_limpa'], RD_te['classe'],
    'V4: Real → Real (desbalanceado)'))
salvar_parcial()

# V5: Sintético → Real (desbalanceado, mesmo test set do V4)
resultados_avancados.append(run_lstm_vision(
    S_tr['frase_limpa'], S_tr['classe'], RD_te['frase_limpa'], RD_te['classe'],
    'V5: Sintético → Real (desbalanceado)'))
salvar_parcial()



=== LSTM | V1: Sintético → Sintético ===
  Vocabulário: 1,690 tokens
  Epoch 1/5 | loss=1.0860
  Epoch 2/5 | loss=1.0060
  Epoch 3/5 | loss=0.8937
  Epoch 4/5 | loss=0.6961
  Epoch 5/5 | loss=0.4417
  ✓ LSTM | F1w=63.38%  F1macro=63.38%  Acc=64.17%  (1s)
  💾 Parcial salvo: c:\Users\Davi\Documentos\tcc-analise-sentimento\resultados\metricas_avancado_movies_partial.csv

=== LSTM | V2: Real → Real ===
  Vocabulário: 3,966 tokens
  Epoch 1/5 | loss=1.1015
  Epoch 2/5 | loss=1.0441
  Epoch 3/5 | loss=0.9867
  Epoch 4/5 | loss=0.9048
  Epoch 5/5 | loss=0.7911
  ✓ LSTM | F1w=38.16%  F1macro=38.16%  Acc=39.17%  (1s)
  💾 Parcial salvo: c:\Users\Davi\Documentos\tcc-analise-sentimento\resultados\metricas_avancado_movies_partial.csv

=== LSTM | V3: Sintético → Real ===
  Vocabulário: 1,690 tokens
  Epoch 1/5 | loss=1.0967
  Epoch 2/5 | loss=1.0060
  Epoch 3/5 | loss=0.8956
  Epoch 4/5 | loss=0.7009
  Epoch 5/5 | loss=0.4547
  ✓ LSTM | F1w=33.88%  F1macro=33.88%  Acc=39.17%  (0s)
  💾 Parcial salvo

In [11]:
# ============================================================
# CÉLULA 14 — BERT nas 5 visões
# ⚠️ ATENÇÃO: V4 (1.2M frases) levará ~6-9 horas na RTX 3070
# ============================================================
# V1
resultados_avancados.append(run_bert_vision(
    S_tr['frase_limpa'], S_tr['classe'], S_te['frase_limpa'], S_te['classe'],
    'V1: Sintético → Sintético'))
salvar_parcial()

# V2
resultados_avancados.append(run_bert_vision(
    R_tr['frase_limpa'], R_tr['classe'], R_te['frase_limpa'], R_te['classe'],
    'V2: Real → Real'))
salvar_parcial()

# V3
resultados_avancados.append(run_bert_vision(
    S_tr['frase_limpa'], S_tr['classe'], R_te['frase_limpa'], R_te['classe'],
    'V3: Sintético → Real'))
salvar_parcial()

# V4 — DEMORA MUITO
print('\n⚠️ INICIANDO V4 com BERT — duração estimada 6-9h na RTX 3070')
resultados_avancados.append(run_bert_vision(
    RD_tr['frase_limpa'], RD_tr['classe'], RD_te['frase_limpa'], RD_te['classe'],
    'V4: Real → Real (desbalanceado)'))
salvar_parcial()

# V5
resultados_avancados.append(run_bert_vision(
    S_tr['frase_limpa'], S_tr['classe'], RD_te['frase_limpa'], RD_te['classe'],
    'V5: Sintético → Real (desbalanceado)'))
salvar_parcial()



=== BERT | V1: Sintético → Sintético ===
  Treino: 480 frases | Teste: 120 frases


ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

## 5. Resultado final — consolidar com clássicos do notebook 03

Lê `resultados/metricas_5_visoes_movies.csv` (gerado pelo notebook 03), concatena com LSTM/BERT e gera tabela + gráfico **5 visões × 5 modelos × 5 métricas**.

In [ ]:
# ============================================================
# CÉLULA 16 — Tabela final + gráficos comparativos
# ============================================================
import matplotlib.pyplot as plt

df_avancado = pd.DataFrame(resultados_avancados)

# Tenta carregar resultados clássicos do notebook 03
csv_classicos = os.path.join(RESULTADOS, 'metricas_5_visoes_movies.csv')
if os.path.exists(csv_classicos):
    df_classicos = pd.read_csv(csv_classicos)
    if 'F1-Macro' not in df_classicos.columns:
        df_classicos['F1-Macro'] = df_classicos['F1-Score']  # fallback antigo
    df_classicos['Tempo (s)'] = None
    df_consolidado = pd.concat([df_classicos, df_avancado], ignore_index=True)
    print(f'Consolidado: {len(df_classicos)} clássicos + {len(df_avancado)} avançados = {len(df_consolidado)} linhas')
else:
    df_consolidado = df_avancado.copy()
    print('AVISO: arquivo de clássicos não encontrado. Apenas LSTM/BERT.')

ordem_visoes = ['V1: Sintético → Sintético', 'V2: Real → Real', 'V3: Sintético → Real',
                'V4: Real → Real (desbalanceado)', 'V5: Sintético → Real (desbalanceado)']
ordem_modelos = ['Naive Bayes', 'Regressão Logística', 'SVM Linear', 'LSTM', 'BERT (Bertimbau)']
df_consolidado['Visão'] = pd.Categorical(df_consolidado['Visão'], categories=ordem_visoes, ordered=True)
df_consolidado['Modelo'] = pd.Categorical(df_consolidado['Modelo'], categories=ordem_modelos, ordered=True)
df_consolidado = df_consolidado.sort_values(['Visão', 'Modelo']).reset_index(drop=True)

# Tabela final F1 weighted e F1 macro
for met in ['F1-Score', 'F1-Macro']:
    pivot = df_consolidado.pivot(index='Visão', columns='Modelo', values=met)[ordem_modelos]
    print(f'\n=== {met} (× 100) ===')
    print((pivot * 100).round(2).to_string())

# Salva CSVs
df_avancado.round(4).to_csv(os.path.join(RESULTADOS, 'metricas_avancado_movies.csv'), index=False)
df_consolidado.round(4).to_csv(os.path.join(RESULTADOS, 'metricas_consolidado_movies.csv'), index=False)
print(f'\n✓ Salvo: resultados/metricas_avancado_movies.csv e metricas_consolidado_movies.csv')

# Gráfico principal: F1 weighted, 5 visões × 5 modelos
pivot_f1 = df_consolidado.pivot(index='Visão', columns='Modelo', values='F1-Score')[ordem_modelos]
fig, ax = plt.subplots(figsize=(15, 7))
pivot_f1.plot(kind='bar', ax=ax, rot=15, edgecolor='black', width=0.85)
ax.set_ylabel('F1-Score (weighted)', fontsize=12)
ax.set_title('Comparativo Clássicos vs Avançados — 5 Visões — Movies', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.legend(title='Modelo', loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=2, fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTADOS, 'grafico_consolidado_movies_f1weighted.png'), dpi=150, bbox_inches='tight')
plt.show()

# Gráfico F1 macro
pivot_fm = df_consolidado.pivot(index='Visão', columns='Modelo', values='F1-Macro')[ordem_modelos]
fig, ax = plt.subplots(figsize=(15, 7))
pivot_fm.plot(kind='bar', ax=ax, rot=15, edgecolor='black', width=0.85)
ax.set_ylabel('F1-Score (macro)', fontsize=12)
ax.set_title('F1 Macro — Clássicos vs Avançados — 5 Visões — Movies', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.legend(title='Modelo', loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=2, fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTADOS, 'grafico_consolidado_movies_f1macro.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Gráficos salvos: grafico_consolidado_movies_f1weighted.png + grafico_consolidado_movies_f1macro.png')
